# Studying the Initial Neutron Star Population

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

## Loading simulation

Select an `initial_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle(
    "../../data/example_simulation_full_sam/initial_population.pkl.gz",
    compression="gzip",
)
data.head()

In [ ]:
age = data["age"]["[yr]"].to_numpy()
r = data["r"]["[kpc]"].to_numpy()
phi = data["phi"]["[rad]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)
vk_r = data["v_r"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
v_phi = (
    data["v_phi"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
vk_z = data["v_z"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
v_orb = (
    data["v_orb"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
vk_phi = v_phi - v_orb
B = data["B"]["[G]"].to_numpy()
chi = data["chi"]["[rad]"].to_numpy()
P = data["P"]["[s]"].to_numpy()
P_dot = data["P_dot"]["[s s^-1]"].to_numpy()

## Age information

Min and max ages:

In [ ]:
print(min(age), max(age))

Mean age of the pulsars

In [ ]:
t_age_mean = np.sum(age) / len(age)
print(t_age_mean)

In [ ]:
cfg["t_age_max"]

## Positional information

Top view of the galactic plane. The distribution plotted here depends on the assumed model for the spatial distribution set up in the configuration file for this simulation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-3.0, 3.0)

plt.show()

Histrograming the pulsars radial position and comparing to the PDF from [Yusifov & Küçük (2004)](https://ui.adsabs.harvard.edu/abs/2004A%26A...422..545Y/abstract).

In [ ]:
def pdf_r(r: float) -> float:
    """
    The Milky Way's stellar radial density in the galactic plane according
    to eq. (15) of Yusifov & Küçük (2004).

    Args:
        r (float): Distance from the galactic center in [kpc].

    Returns:
        float: Stellar radial density in [1/kpc].
    """

    # Here we keep R_sun = 8.5 kpc for consistency with the results
    # of Yusifov & Küçük (2004).
    rsun = 8.5  # Sun's distance from the galactic center in [kpc].
    A = 37.6  # +- 1.90 [1/kpc^2]
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    r1 = 0.55  # +- 0.10 [kpc]

    # Stellar surface density following eq. (15) of Yusifov & Küçük (2004).
    rho = (
        A
        * ((r + r1) / (rsun + r1)) ** a
        * np.exp(-b * (r - rsun) / (rsun + r1))
    )

    # Multiply the stellar surface density with the area element in polar coordinates.
    pdf_r = 2 * np.pi * r * rho

    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve.

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_edges = np.linspace(0.0, 30.0, 51)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    r,
    bins=r_edges,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Initial simulation",
    density=True,
)
ax.plot(
    r_edges,
    pdf_r(r_edges) / pdf_area,
    linestyle="-",
    lw=4,
    color="black",
    alpha=1,
    label="Theoretical YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Normalized radial PDF")
plt.xlim(0.0, 30.0)
plt.legend(frameon=False, loc=1)

plt.show()

Histrograming the pulsars $z$ position and comparing to underlying PDF.

In [ ]:
def pdf_z(z: float) -> float:
    """
    Probability density function for the height from the galactic equatorial plane
    according to eq. (2) in Gullon et al. (2014).

    Args:
        z (float): Distance from the galactic plane in [kpc].

    Returns:
        float: Distribution of stars per kpc in z direction.
    """

    # We use an exponential distribution as given by Wainscoat et al. (1992)
    # and choose a mean scale height characteristic for a young distribution as
    # obtained by Gullon et al. (2014).

    h_c = 0.18
    pdf_z = 1.0 / h_c * np.exp(-z / h_c)

    return pdf_z

In [ ]:
z_edges = np.linspace(0.0, 2.0, 51)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    z,
    bins=z_edges,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Initial simulation",
    density=True,
)
ax.plot(
    z_edges,
    pdf_z(z_edges),
    linestyle="-",
    lw=4,
    color="black",
    alpha=1,
    label="Theoretical exp",
)
plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Normalized height PDF")
plt.xlim(0.0, 2.0)
plt.legend(frameon=False, loc=1)

plt.show()

## Proper velocity information

Histogramed kick velocity components.

In [ ]:
vk_edges = np.linspace(-1500.0, 1500.0, 51)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    vk_r,
    bins=vk_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}r}$",
)
ax.hist(
    vk_phi,
    bins=vk_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}\phi}$",
)
ax.hist(
    vk_z,
    bins=vk_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}z}$",
)
ax.set_xlabel(r"Kick velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Distribution of the total kick velocity magnitude.

In [ ]:
def pdf_kick_velocity_maxwell(v: float) -> float:
    """
    Maxwell probability density function for the neutron stars' initial kick
    velocity magnitude following Hobbs et al. (2005).

    Args:
        v (float): Initial kick velocity magnitude in [km/s].

    Returns:
        float: Stellar kick velocity distribution in [1/(km/s)].
    """
    sigma = 265.0
    pdf_vk = (
        np.sqrt(2 / np.pi)
        * v**2
        / (sigma**3)
        * np.exp(-(v**2) / (2 * sigma**2))
    )

    return pdf_vk

In [ ]:
vk_tot = np.sqrt(vk_r**2 + vk_phi**2 + vk_z**2)

vk_edges = np.linspace(0, 1500.0, 51)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    vk_tot,
    bins=vk_edges,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Simulated",
    density=True,
)
ax.plot(
    vk_edges,
    pdf_kick_velocity_maxwell(vk_edges),
    linestyle="-",
    lw=4,
    color="black",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"Kick velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=1)

plt.show()

Orbital (angular) velocity due to Galactic potential.

In [ ]:
r = np.sqrt(x**2 + y**2)

fig, ax = plt.subplots(figsize=(15, 8))

scatter = ax.scatter(
    r,
    abs(v_orb),
    linestyle="None",
    marker="o",
    c=abs(z),
    cmap="jet",
    s=1,
    alpha=1,
    rasterized=True,
)
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label(r"$|z|$ [kpc]")

ax.set_xlabel(r"$r$ [kpc]")
ax.set_ylabel(r"$v_{\rm orb}$ [km s$^{-1}$]")

plt.show()

Histogramed velocity components.

In [ ]:
v_edges = np.linspace(-1500.0, 1500.0, 51)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    vk_r,
    bins=v_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_r$",
)
ax.hist(
    vk_phi + v_orb,
    bins=v_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{\phi}$",
)
ax.hist(
    vk_z,
    bins=v_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_z$",
)
ax.set_xlabel(r"Velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Distribution of the total velocity magnitude.

In [ ]:
v_tot = np.sqrt(vk_r**2 + (vk_phi + v_orb) ** 2 + vk_z**2)

v_edges = np.linspace(0.0, 1500.0, 51)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    v_tot,
    bins=v_edges,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Simulated",
    density=True,
)
ax.plot(
    v_edges,
    pdf_kick_velocity_maxwell(v_edges),
    linestyle="-",
    lw=4,
    color="black",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"3D velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

## Magneto-rotational information

Histograming the periods, magnetic fields and misalignment angles.

In [ ]:
def Gaussian(x, mean, sigma):
    y = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mean) ** 2) / (2 * sigma**2))
    )
    return y

In [ ]:
P_bins = np.linspace(-0.5, 2.0, 101)
print(max(P), min(P))

In [ ]:
cfg["P_initial_mean"] = 0.3
cfg["P_initial_sigma"] = 0.2

In [ ]:
pdf_P_initial_area = quad(
    Gaussian, 0, 100, args=(cfg["P_initial_mean"], cfg["P_initial_sigma"])
)[0]
print(pdf_P_initial_area)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P,
    bins=P_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulation",
    density=True,
)
ax.plot(
    P_bins,
    Gaussian(P_bins, cfg["P_initial_mean"], cfg["P_initial_sigma"]),
    linestyle="--",
    lw=4,
    color="black",
    label="Theoretical",
)
ax.plot(
    P_bins[P_bins > 0],
    Gaussian(P_bins[P_bins > 0], cfg["P_initial_mean"], cfg["P_initial_sigma"])
    / pdf_P_initial_area,
    linestyle="-",
    lw=4,
    color="black",
    label="Rescaled",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Normalized PDF")
plt.xlim(-0.5, 2.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
B_log10_bins = np.linspace(9.0, 17.0, 101)
print(max(np.log10(B)), min(np.log10(B)))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    np.log10(B),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulation",
    density=True,
)
ax.plot(
    B_log10_bins,
    Gaussian(
        B_log10_bins, cfg["B_initial_log10_mean"], cfg["B_initial_log10_sigma"]
    ),
    linestyle="-",
    lw=4,
    color="black",
    label="Theoretical",
)
plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"Normalized PDF")
plt.xlim(9.0, 17.0)
plt.legend(frameon=False, loc=0)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 101)
print(max(chi), min(chi))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    chi,
    bins=chi_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulation",
    density=True,
)
ax.plot(
    chi_bins,
    np.sin(chi_bins),
    linestyle="-",
    lw=4,
    color="black",
    label="Theoretical",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Normalized PDF")
plt.xlim(0.0, np.pi / 2)
plt.legend(frameon=False, loc=0)

plt.show()

Plotting the $P-B$ distribution of the initial pulsar population.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.loglog(P, B, "o", color="darkgray", ms=1, alpha=0.1)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$B$ [G]")

plt.show()

Plotting the $P-\dot{P}$ diagram of the initial pulsar population.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.loglog(P, P_dot / const.YR_TO_S, "o", color="darkgray", ms=1, alpha=0.1)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")

plt.show()

Note that this is not a snapshot in time of the pulsar population, but all initial parameter sets projected into the $P-\dot{P}$ plane.